# Core v2 — downsampling, баланс human↔seen_llm, train / val / test + eval slices

Сборка по `dataset_design_final.md` / `dataset_contract.md`: matched `scenario_family`, Claude **только** в test, `financial_qa` только полные пары. **Конфиг** — первая code-ячейка.

**Main comparable pool (train, val, test_seen):** только **human + seen_llm** (OpenAI/Mistral; для `financial_qa` — HC3 ChatGPT как seen). Бюджет на сторону: `main_seen_budget_per_side = min(target_cap_family, n_human_raw, n_seen_llm_raw)` — **Claude не входит** в этот баланс.

**Claude holdout:** отдельный add-on: `claude_holdout_budget = min(round(main_seen_budget * CLAUDE_HOLDOUT_REL_FRAC), n_claude_raw)` (дефолт **0.25**). Все эти строки → `core_eval_slice=test_claude_holdout`, **не** в train/val/test_seen.

**Сплит 75/15/10** применяется **только** к main pool; затем `test_full = test_seen ∪ test_claude_only` (`core_test.jsonl` = test_full).

**Файлы test:** `core_test_full.jsonl`, `core_test_non_claude.jsonl` (test_seen), `core_test_claude_only.jsonl`.

**Диагностика:** `v2/docs/core_split_diagnostics.md`, `v2/data/interim/assembled/core_split_diagnostics.json`, `v2/outputs/tables/core_crosstab_*.csv`.


In [ ]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

def resolve_v2_base() -> Path:
    """Работает из корня репо или из v2/ (ищет v2/pyproject.toml вверх по родителям)."""
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        cand = p / "v2" / "pyproject.toml"
        if cand.is_file():
            return p / "v2"
        if p.name == "v2" and (p / "pyproject.toml").is_file():
            return p
    raise FileNotFoundError("Запустите ноутбук из репозитория (нужен v2/pyproject.toml).")


BASE = resolve_v2_base()
sys.path.insert(0, str(BASE))

ASSEMBLED = BASE / "data" / "interim" / "assembled"
TABLES_DIR = BASE / "outputs" / "tables"
DOCS_DIR = BASE / "docs"

PATH_DS1 = ASSEMBLED / "dataset1_human.jsonl"
PATH_DS2 = ASSEMBLED / "dataset2_llm.jsonl"
PATH_CORE_FULL = ASSEMBLED / "core_v2.jsonl"
PATH_MANIFEST = ASSEMBLED / "core_manifest.json"
PATH_DESC = DOCS_DIR / "core_dataset_description.md"
PATH_DIAG_MD = DOCS_DIR / "core_split_diagnostics.md"
PATH_DIAG_JSON = ASSEMBLED / "core_split_diagnostics.json"

RANDOM_SEED = 42
TRAIN_FRAC = 0.75
VAL_FRAC = 0.15
TEST_FRAC = 0.10

TARGET_N_PER_SIDE: dict[str, int | None] = {
    "phishing_email": 3000,
    "advance_fee_scam_email": 1800,
    "fraud_sms_deceptive": None,
    "legitimate_email": 3000,
    "legitimate_sms": None,
}
FRAUD_SMS_MAX_PER_SIDE = 1200
LEGITIMATE_SMS_TARGET_CAP = 1200

TARGET_PAIRS_FQ = 800
CLAUDE_HOLDOUT_REL_FRAC = 0.25

SCENARIO_NON_FQ: tuple[str, ...] = (
    "advance_fee_scam_email",
    "fraud_sms_deceptive",
    "legitimate_email",
    "legitimate_sms",
    "phishing_email",
)

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9
assert 0.20 <= CLAUDE_HOLDOUT_REL_FRAC <= 0.30
assert set(SCENARIO_NON_FQ) == set(TARGET_N_PER_SIDE.keys())

REQUIRED_KEYS = frozenset(
    {
        "text",
        "label",
        "label_str",
        "fraudness",
        "channel",
        "scenario_family",
        "source_family",
        "dataset_source",
        "time_band",
        "length_bin",
        "origin_model",
        "split",
        "core_eval_slice",
    }
)

if not PATH_DS1.is_file() or not PATH_DS2.is_file():
    raise FileNotFoundError(f"Need {PATH_DS1} and {PATH_DS2} (run notebooks 17 and 18 first).")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
ASSEMBLED.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(np.random.PCG64(RANDOM_SEED))

print("BASE:", BASE)
print("TARGET_N_PER_SIDE:", TARGET_N_PER_SIDE)
print("FRAUD_SMS_MAX_PER_SIDE:", FRAUD_SMS_MAX_PER_SIDE, "LEGITIMATE_SMS_TARGET_CAP:", LEGITIMATE_SMS_TARGET_CAP)
print("TARGET_PAIRS_FQ:", TARGET_PAIRS_FQ, "CLAUDE_HOLDOUT_REL_FRAC:", CLAUDE_HOLDOUT_REL_FRAC)
print("split:", TRAIN_FRAC, VAL_FRAC, TEST_FRAC)


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


rows1 = load_jsonl(PATH_DS1)
rows2 = load_jsonl(PATH_DS2)
for i, r in enumerate(rows1):
    r["_ds"] = "dataset1_human"
    r["_src_line"] = i + 1
for i, r in enumerate(rows2):
    r["_ds"] = "dataset2_llm"
    r["_src_line"] = i + 1

df_h = pd.DataFrame(rows1)
df_l = pd.DataFrame(rows2)

raw_counts: dict = {
    "dataset1_total": len(df_h),
    "dataset2_total": len(df_l),
    "human_by_family": df_h.groupby("scenario_family").size().to_dict(),
    "llm_by_family": df_l.groupby("scenario_family").size().to_dict(),
}

print("dataset1:", len(df_h), "dataset2:", len(df_l))

In [ ]:
def stratified_downsample(
    df: pd.DataFrame, n: int, strata_cols: list[str], g: np.random.Generator
) -> pd.DataFrame:
    """Без возврата; пропорции по стратам (largest remainder)."""
    df = df.reset_index(drop=True)
    if n <= 0 or len(df) == 0:
        return df.iloc[0:0].copy()
    if len(df) <= n:
        return df.copy()
    use_cols = [c for c in strata_cols if c in df.columns]
    df = df.copy()
    if not use_cols:
        df["_one"] = "x"
        use_cols = ["_one"]
    df["_sk"] = df[use_cols].astype(str).agg("|".join, axis=1)
    vc = df["_sk"].value_counts()
    total = len(df)
    keys = vc.index.tolist()
    quota = {k: int(np.floor(n * float(vc.loc[k]) / total)) for k in keys}
    rem = n - sum(quota.values())
    frac_rem = sorted(
        [((n * float(vc.loc[k]) / total) - quota[k], k) for k in keys],
        key=lambda x: -x[0],
    )
    for _, k in frac_rem:
        if rem <= 0:
            break
        if quota[k] < int(vc.loc[k]):
            quota[k] += 1
            rem -= 1
    while rem > 0:
        progressed = False
        for k in keys:
            if rem <= 0:
                break
            if quota[k] < int(vc.loc[k]):
                quota[k] += 1
                rem -= 1
                progressed = True
        if not progressed:
            break
    parts: list[pd.DataFrame] = []
    for k, q in quota.items():
        if q <= 0:
            continue
        sub = df[df["_sk"] == k]
        take = min(q, len(sub))
        parts.append(
            sub.sample(n=take, random_state=int(g.integers(0, 2**31 - 1)))
        )
    out = pd.concat(parts, ignore_index=True)
    out = out.drop(columns=["_sk", "_one"], errors="ignore")
    if len(out) > n:
        out = out.sample(
            n=n, random_state=int(g.integers(0, 2**31 - 1)), ignore_index=True
        )
    return out


def human_strata_cols(sf: str, h_sub: pd.DataFrame) -> list[str]:
    if sf == "legitimate_email":
        return ["length_bin", "source_family"]
    if sf == "fraud_sms_deceptive":
        return ["length_bin", "source_family"]
    cols = ["length_bin"]
    if sf in ("phishing_email", "advance_fee_scam_email") and "time_band" in h_sub.columns:
        if h_sub["time_band"].nunique(dropna=False) > 1:
            cols = ["length_bin", "time_band"]
    return cols


def llm_seen_strata_cols(l_sub: pd.DataFrame) -> list[str]:
    cols = ["length_bin", "generator_lane"]
    if "origin_model" in l_sub.columns and l_sub["origin_model"].nunique(dropna=False) > 1:
        cols = ["length_bin", "generator_lane", "origin_model"]
    return cols


def split_seen_openai_mistral(seen: pd.DataFrame, n_budget: int, g: np.random.Generator):
    oa = seen[seen["generator_lane"] == "seen_openai"]
    mi = seen[seen["generator_lane"] == "seen_mistral"]
    wo, wm = len(oa), len(mi)
    if n_budget <= 0 or wo + wm == 0:
        return oa.iloc[0:0], mi.iloc[0:0]
    n_oa = int(round(n_budget * wo / (wo + wm))) if (wo + wm) else 0
    n_oa = min(max(0, n_oa), wo, n_budget)
    n_mi = min(wm, n_budget - n_oa)
    if n_oa + n_mi < n_budget:
        slack = n_budget - n_oa - n_mi
        add = min(slack, wo - n_oa)
        n_oa += add
        n_mi += min(slack - add, wm - n_mi)
    oa_s = stratified_downsample(oa, n_oa, ["length_bin"], g) if n_oa else oa.iloc[0:0]
    mi_s = stratified_downsample(mi, n_mi, ["length_bin"], g) if n_mi else mi.iloc[0:0]
    return oa_s, mi_s


downsampling_report: dict = {"per_family": {}, "financial_qa": {}}
main_seen_budget_per_family: dict[str, int] = {}
claude_holdout_budget_per_family: dict[str, int] = {}
balanced_chunks: list[pd.DataFrame] = []

for sf in SCENARIO_NON_FQ:
    h_raw = df_h[df_h["scenario_family"] == sf]
    l_raw = df_l[df_l["scenario_family"] == sf]
    cl_m = (l_raw["generator_lane"] == "holdout_claude") | (
        l_raw["source_family"] == "llm_holdout_claude"
    )
    cl_pool = l_raw[cl_m]
    seen_pool = l_raw[~cl_m]

    if sf == "fraud_sms_deceptive":
        target_cap = FRAUD_SMS_MAX_PER_SIDE
    elif sf == "legitimate_sms":
        target_cap = LEGITIMATE_SMS_TARGET_CAP
    else:
        req = TARGET_N_PER_SIDE[sf]
        target_cap = len(h_raw) if req is None else int(req)

    n_h_raw = int(len(h_raw))
    n_seen_raw = int(len(seen_pool))
    n_cl_raw = int(len(cl_pool))

    main_seen_budget = min(target_cap, n_h_raw, n_seen_raw)
    claude_holdout_budget = min(
        int(round(main_seen_budget * CLAUDE_HOLDOUT_REL_FRAC)), n_cl_raw
    )
    main_seen_budget_per_family[sf] = main_seen_budget
    claude_holdout_budget_per_family[sf] = claude_holdout_budget

    h_s = stratified_downsample(
        h_raw, main_seen_budget, human_strata_cols(sf, h_raw), rng
    )
    oa_s, mi_s = split_seen_openai_mistral(seen_pool, main_seen_budget, rng)
    seen_s = pd.concat([oa_s, mi_s], ignore_index=True)
    if len(seen_s) != main_seen_budget:
        raise RuntimeError(
            f"{sf}: seen_llm sample size {len(seen_s)} != main_seen_budget {main_seen_budget}"
        )
    cl_s = (
        stratified_downsample(cl_pool, claude_holdout_budget, ["length_bin"], rng)
        if claude_holdout_budget
        else cl_pool.iloc[0:0]
    )

    req = TARGET_N_PER_SIDE[sf]
    want_side = (
        FRAUD_SMS_MAX_PER_SIDE
        if sf == "fraud_sms_deceptive"
        else LEGITIMATE_SMS_TARGET_CAP
        if sf == "legitimate_sms"
        else (len(h_raw) if req is None else int(req))
    )
    n_oa_raw = int((seen_pool["generator_lane"] == "seen_openai").sum())
    n_mi_raw = int((seen_pool["generator_lane"] == "seen_mistral").sum())
    human_sf_n = int(h_raw["source_family"].nunique(dropna=False))

    downsampling_report["per_family"][sf] = {
        "n_human_raw": n_h_raw,
        "n_seen_llm_raw": n_seen_raw,
        "n_claude_raw": n_cl_raw,
        "n_seen_openai_raw": n_oa_raw,
        "n_seen_mistral_raw": n_mi_raw,
        "human_source_families_raw_n": human_sf_n,
        "target_cap_family": int(target_cap),
        "main_seen_budget_per_side": int(main_seen_budget),
        "claude_holdout_budget": int(claude_holdout_budget),
        "n_human_main_selected": int(len(h_s)),
        "n_seen_llm_main_selected": int(len(seen_s)),
        "n_claude_selected": int(len(cl_s)),
        "capped_by_availability_human": bool(
            sf not in ("fraud_sms_deceptive", "legitimate_sms")
            and req is not None
            and n_h_raw < int(req)
        ),
        "capped_by_seen_llm_vs_target": bool(main_seen_budget < target_cap),
        "capped_by_claude_availability": bool(
            n_cl_raw < int(round(main_seen_budget * CLAUDE_HOLDOUT_REL_FRAC))
        ),
    }

    balanced_chunks.append(h_s)
    balanced_chunks.append(seen_s)
    balanced_chunks.append(cl_s)

fh = df_h[df_h["scenario_family"] == "financial_qa"].copy()
fl = df_l[df_l["scenario_family"] == "financial_qa"].copy()
h_by = fh.groupby("question_id").size()
l_by = fl.groupby("question_id").size()
complete = sorted(set(h_by.index[h_by == 1]) & set(l_by.index[l_by == 1]))
meta_rows = []
for q in complete:
    hr = fh[fh["question_id"] == q].iloc[0]
    meta_rows.append({"question_id": q, "length_bin": hr["length_bin"]})
meta_df = pd.DataFrame(meta_rows)
n_pairs = min(TARGET_PAIRS_FQ, len(meta_df))
meta_s = stratified_downsample(meta_df, n_pairs, ["length_bin"], rng)
sel_q = set(meta_s["question_id"].tolist())
fq_parts: list[pd.DataFrame] = []
for q in sorted(sel_q):
    fq_parts.append(fh[fh["question_id"] == q])
    fq_parts.append(fl[fl["question_id"] == q])
fq_df = pd.concat(fq_parts, ignore_index=True) if fq_parts else fh.iloc[0:0]

downsampling_report["financial_qa"] = {
    "complete_pairs_available": len(complete),
    "pairs_target": TARGET_PAIRS_FQ,
    "pairs_selected": n_pairs,
    "rows_written": int(len(fq_df)),
}
balanced_chunks.append(fq_df)

df = pd.concat(balanced_chunks, ignore_index=True)
df["core_row_id"] = range(len(df))
df["split"] = pd.NA

print("Core rows after downsampling:", len(df))
print(df.groupby(["scenario_family", "label"]).size().unstack(fill_value=0))


In [ ]:
def split_sizes(n: int) -> tuple[int, int, int]:
    """Те же целочисленные квоты, что и в labels_for_n (для симметрии human / seen_llm)."""
    if n <= 0:
        return (0, 0, 0)
    if n == 1:
        return (1, 0, 0)
    n_tr = int(n * TRAIN_FRAC)
    n_va = int(n * VAL_FRAC)
    n_te = n - n_tr - n_va
    if n_tr == 0:
        n_tr = 1
        n_te = max(0, n - n_tr - n_va)
    if n_va == 0 and n >= 3:
        n_va = 1
        n_te = n - n_tr - n_va
    if n_te < 0:
        n_te = 0
        n_va = max(0, n - n_tr)
    while n_tr + n_va + n_te < n:
        n_tr += 1
    while n_tr + n_va + n_te > n:
        if n_te > 0:
            n_te -= 1
        elif n_va > 0:
            n_va -= 1
        else:
            n_tr -= 1
    return (n_tr, n_va, n_te)


def three_way_stratified_assign(
    df: pd.DataFrame,
    idx: list[int],
    n_tr: int,
    n_va: int,
    n_te: int,
    strata_cols: list[str],
    g: np.random.Generator,
) -> tuple[list[int], list[int], list[int]]:
    if not idx:
        return [], [], []
    sub = df.loc[idx].copy()
    sub["_rid"] = sub.index
    sub = sub.reset_index(drop=True)
    tr_df = stratified_downsample(sub, n_tr, strata_cols, g)
    tr_ids = set(tr_df["_rid"].tolist())
    rem = sub[~sub["_rid"].isin(tr_ids)]
    va_df = stratified_downsample(rem, n_va, strata_cols, g)
    va_ids = set(va_df["_rid"].tolist())
    te_df = rem[~rem["_rid"].isin(va_ids)]
    if len(te_df) != n_te:
        raise RuntimeError(
            f"three_way_stratified_assign: n_te mismatch got {len(te_df)} want {n_te} (n={len(idx)})"
        )
    return (
        tr_df["_rid"].tolist(),
        va_df["_rid"].tolist(),
        te_df["_rid"].tolist(),
    )


def labels_for_n(n: int, g: np.random.Generator) -> list[str]:
    if n <= 0:
        return []
    if n == 1:
        return ["train"]
    n_tr, n_va, n_te = split_sizes(n)
    labs = ["train"] * n_tr + ["val"] * n_va + ["test"] * n_te
    order = g.permutation(n)
    out = [None] * n
    for i in range(n):
        out[int(order[i])] = labs[i]
    return out


def assign_splits(indices: list[int], g: np.random.Generator) -> None:
    labs = labels_for_n(len(indices), g)
    for idx, lab in zip(indices, labs):
        df.at[idx, "split"] = lab


manifest_log: dict = {"stratum_merges": [], "fq_warnings": []}


In [ ]:
lane = df["generator_lane"].fillna("")
sff = df["source_family"].fillna("")
claude_m = (lane == "holdout_claude") | (sff == "llm_holdout_claude")
df.loc[claude_m, "split"] = "test"
print("Claude holdout → test:", int(claude_m.sum()))

fq_m = df["scenario_family"] == "financial_qa"
fq_idx = df.index[fq_m & df["split"].isna()].tolist()
fq_df = df.loc[fq_idx].copy()

strata_qids: dict[str, list] = defaultdict(list)
for qid, grp in fq_df.groupby("question_id"):
    gidx = grp.index.tolist()
    human = grp[grp["label"] == 0]
    lb = str(human.iloc[0]["length_bin"]) if len(human) else str(grp.iloc[0]["length_bin"])
    strata_qids[lb].append((qid, gidx))

for lb, pairs in sorted(strata_qids.items()):
    qids_order = [p[0] for p in pairs]
    rng.shuffle(qids_order)
    labs = labels_for_n(len(qids_order), rng)
    qid_to_lab = dict(zip(qids_order, labs))
    for qid, gidx in pairs:
        lab = qid_to_lab[qid]
        for ix in gidx:
            df.at[ix, "split"] = lab

print("financial_qa rows:", int(fq_m.sum()))


In [ ]:
for sf in SCENARIO_NON_FQ:
    idx_h = df.index[(df["scenario_family"] == sf) & (df["label"] == 0)].tolist()
    idx_s = df.index[
        (df["scenario_family"] == sf) & (df["label"] == 1) & (~claude_m)
    ].tolist()
    nh, ns = len(idx_h), len(idx_s)
    if nh != ns:
        raise ValueError(f"{sf}: human {nh} vs seen_llm {ns} (ожидался баланс после downsampling)")
    if nh == 0:
        continue
    n_tr, n_va, n_te = split_sizes(nh)
    hcols = human_strata_cols(sf, df.loc[idx_h])
    lcols = llm_seen_strata_cols(df.loc[idx_s])
    th, vh, teh = three_way_stratified_assign(df, idx_h, n_tr, n_va, n_te, hcols, rng)
    ts, vs, tes = three_way_stratified_assign(df, idx_s, n_tr, n_va, n_te, lcols, rng)
    df.loc[th, "split"] = "train"
    df.loc[ts, "split"] = "train"
    df.loc[vh, "split"] = "val"
    df.loc[vs, "split"] = "val"
    df.loc[teh, "split"] = "test"
    df.loc[tes, "split"] = "test"

assert not df["split"].isna().any()
assert set(df["split"].unique()) <= {"train", "val", "test"}
print("non-FQ families split: symmetric human / seen_llm per family")

df["core_eval_slice"] = pd.NA
df.loc[df["split"] == "train", "core_eval_slice"] = "train"
df.loc[df["split"] == "val", "core_eval_slice"] = "val"
df.loc[claude_m, "core_eval_slice"] = "test_claude_holdout"
df.loc[(df["split"] == "test") & (~claude_m), "core_eval_slice"] = "test_seen"
assert not df["core_eval_slice"].isna().any()


In [ ]:
validation: dict = {"checks": []}

bad_keys = 0
for _, r in df.iterrows():
    miss = REQUIRED_KEYS - set(r.index)
    if miss:
        bad_keys += 1
validation["checks"].append({"name": "required_keys", "ok": bad_keys == 0, "bad_rows": bad_keys})

cl_in_tv = df[claude_m & df["split"].isin(["train", "val"])]
validation["checks"].append(
    {
        "name": "claude_only_test",
        "ok": len(cl_in_tv) == 0,
        "violations": int(len(cl_in_tv)),
    }
)

fq_pair_viol = 0
fq_detail = []
for qid, grp in df[df["scenario_family"] == "financial_qa"].groupby("question_id"):
    if len(grp) != 2:
        fq_pair_viol += 1
        fq_detail.append({"question_id": int(qid) if pd.notna(qid) else None, "n": len(grp)})
        continue
    if len(grp["split"].unique()) != 1:
        fq_pair_viol += 1
        fq_detail.append({"question_id": int(qid), "splits": grp["split"].unique().tolist()})
    if set(int(x) for x in grp["label"].tolist()) != {0, 1}:
        fq_pair_viol += 1
validation["checks"].append(
    {"name": "financial_qa_complete_pairs", "ok": fq_pair_viol == 0, "pair_violations": fq_pair_viol}
)
validation["fq_detail_sample"] = fq_detail[:10]

bal_whole_ok = True
bal_whole_issues: list[dict] = []
for sf in SCENARIO_NON_FQ:
    sub = df[df["scenario_family"] == sf]
    h = int((sub["label"] == 0).sum())
    s = int(((sub["label"] == 1) & (~claude_m)).sum())
    if h != s:
        bal_whole_ok = False
        bal_whole_issues.append({"scenario_family": sf, "human": h, "seen_llm": s})
validation["checks"].append(
    {
        "name": "balance_human_seenllm_whole_per_family",
        "ok": bal_whole_ok,
        "mismatches": bal_whole_issues,
    }
)

def _balance_tol(nh: int, nl: int) -> int:
    """Строго 0; ±1 только при совсем маленьких срезах (округление крошечных страт)."""
    if nh + nl <= 8:
        return 1
    return 0



def _slice_balance_issues(es_list: list[str]) -> list[dict]:
    issues: list[dict] = []
    for sf in list(SCENARIO_NON_FQ) + ["financial_qa"]:
        for es in es_list:
            sub = df[(df["scenario_family"] == sf) & (df["core_eval_slice"] == es)]
            nh = int((sub["label"] == 0).sum())
            nl = int((sub["label"] == 1).sum())
            if abs(nh - nl) > _balance_tol(nh, nl):
                issues.append(
                    {
                        "scenario_family": sf,
                        "core_eval_slice": es,
                        "n_human": nh,
                        "n_llm": nl,
                        "diff": abs(nh - nl),
                    }
                )
    return issues

main_issues = _slice_balance_issues(["train", "val", "test_seen"])
validation["checks"].append(
    {
        "name": "balance_human_seenllm_per_family_main_slices",
        "ok": len(main_issues) == 0,
        "tolerance_rule": "0 except nh+nl<=8 -> 1",
        "mismatches": main_issues[:80],
    }
)

ts_issues = _slice_balance_issues(["test_seen"])
validation["checks"].append(
    {
        "name": "balance_human_seenllm_per_family_test_seen",
        "ok": len(ts_issues) == 0,
        "tolerance_rule": "0 except nh+nl<=8 -> 1",
        "mismatches": ts_issues[:80],
    }
)

test_seen_df = df[(df["split"] == "test") & (~claude_m)]
test_claude_df = df[(df["split"] == "test") & claude_m]
test_df = df[df["split"] == "test"]
validation["test_counts"] = {
    "test_full": int(len(test_df)),
    "test_non_claude_rows": int(len(test_seen_df)),
    "test_claude_only_rows": int(len(test_claude_df)),
}

empty_text = int((df["text"].fillna("").astype(str).str.strip() == "").sum())
validation["checks"].append(
    {"name": "no_empty_text", "ok": empty_text == 0, "empty_rows": empty_text}
)

_norm_t = df["text"].fillna("").astype(str).str.strip()
_dup = int(_norm_t.duplicated().sum())
validation["checks"].append(
    {"name": "no_duplicate_text_global", "ok": _dup == 0, "duplicate_rows": _dup}
)

_fq_shape = 0
for qid, grp in df[df["scenario_family"] == "financial_qa"].groupby("question_id"):
    if len(grp) != 2:
        _fq_shape += 1
validation["checks"].append(
    {
        "name": "financial_qa_question_id_pair_shape",
        "ok": _fq_shape == 0,
        "bad_question_groups": _fq_shape,
    }
)

validation["checks"].append(
    {
        "name": "eval_slices_present",
        "ok": set(df["core_eval_slice"].dropna().unique())
        >= {"train", "val", "test_seen", "test_claude_holdout"},
        "slices": df["core_eval_slice"].value_counts(dropna=False).to_dict(),
    }
)

_ct_ch_sp_lab = pd.crosstab([df["channel"], df["split"]], df["label"])
_ch_lab_issues: list[str] = []
for ch in df["channel"].unique():
    for sp in ["train", "val", "test"]:
        sub = _ct_ch_sp_lab.loc[(ch, sp)] if (ch, sp) in _ct_ch_sp_lab.index else None
        if sub is None or sub.sum() == 0:
            continue
        n0, n1 = int(sub.get(0, 0)), int(sub.get(1, 0))
        if min(n0, n1) == 0 and max(n0, n1) > 50:
            _ch_lab_issues.append(f"{ch}/{sp}: label0={n0} label1={n1}")
validation["checks"].append(
    {
        "name": "audit_channel_split_label",
        "ok": len(_ch_lab_issues) == 0,
        "warnings": _ch_lab_issues[:20],
    }
)

_om_issues: list[str] = []
for sf in SCENARIO_NON_FQ:
    rep = downsampling_report["per_family"][sf]
    oa_r = int(rep.get("n_seen_openai_raw", 0))
    mi_r = int(rep.get("n_seen_mistral_raw", 0))
    if oa_r > 0 and mi_r > 0:
        sub = df[(df["scenario_family"] == sf) & (df["label"] == 1) & (~claude_m)]
        c_oa = int((sub["generator_lane"] == "seen_openai").sum())
        c_mi = int((sub["generator_lane"] == "seen_mistral").sum())
        if c_oa == 0 or c_mi == 0:
            _om_issues.append(
                f"{sf}: raw had both lanes (oa={oa_r}, mi={mi_r}) but core seen_llm oa={c_oa} mi={c_mi}"
            )
    sub = df[(df["scenario_family"] == sf) & (df["label"] == 1) & (~claude_m)]
    if len(sub) >= 10:
        vc = sub["origin_model"].fillna("(na)").value_counts()
        if len(vc) == 1 and vc.iloc[0] == len(sub):
            _om_issues.append(f"{sf}: single origin_model for all seen_llm rows")
validation["checks"].append(
    {
        "name": "origin_model_distribution_check",
        "ok": len(_om_issues) == 0,
        "warnings": _om_issues,
    }
)

_sf_issues: list[str] = []
for sf in ("legitimate_email", "fraud_sms_deceptive"):
    rep = downsampling_report["per_family"][sf]
    if int(rep.get("human_source_families_raw_n", 1)) <= 1:
        continue
    hsub = df[(df["scenario_family"] == sf) & (df["label"] == 0)]
    nu = int(hsub["source_family"].nunique(dropna=False))
    if nu < 2:
        _sf_issues.append(f"{sf}: human raw had multiple source_family but selected human nu={nu}")
validation["checks"].append(
    {
        "name": "source_family_distribution_check",
        "ok": len(_sf_issues) == 0,
        "warnings": _sf_issues,
    }
)

for c in validation["checks"]:
    print(c)

_hard = {
    "required_keys",
    "claude_only_test",
    "financial_qa_complete_pairs",
    "balance_human_seenllm_whole_per_family",
    "balance_human_seenllm_per_family_main_slices",
    "balance_human_seenllm_per_family_test_seen",
    "no_empty_text",
    "no_duplicate_text_global",
    "financial_qa_question_id_pair_shape",
    "eval_slices_present",
    "origin_model_distribution_check",
    "source_family_distribution_check",
}
if not all(c["ok"] for c in validation["checks"] if c["name"] in _hard):
    raise ValueError("Validation failed")
if bad_keys:
    raise ValueError("Missing required keys")


In [ ]:
run_ts = datetime.now(timezone.utc).isoformat()

out_records = df.drop(columns=["_ds", "_src_line"], errors="ignore").to_dict("records")

with PATH_CORE_FULL.open("w", encoding="utf-8") as f:
    for rec in out_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def _write_jsonl(path: Path, recs: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


for sp in ("train", "val"):
    sub = [r for r in out_records if r.get("split") == sp]
    outp = ASSEMBLED / f"core_{sp}.jsonl"
    _write_jsonl(outp, sub)
    print(sp, len(sub), outp)

test_full_recs = [r for r in out_records if r.get("split") == "test"]
test_nc_recs = [r for r in out_records if r.get("core_eval_slice") == "test_seen"]
test_cl_recs = [r for r in out_records if r.get("core_eval_slice") == "test_claude_holdout"]
_write_jsonl(ASSEMBLED / "core_test_full.jsonl", test_full_recs)
_write_jsonl(ASSEMBLED / "core_test_non_claude.jsonl", test_nc_recs)
_write_jsonl(ASSEMBLED / "core_test_claude_only.jsonl", test_cl_recs)
_write_jsonl(ASSEMBLED / "core_test.jsonl", test_full_recs)
print("test_full", len(test_full_recs), ASSEMBLED / "core_test_full.jsonl")
print("test_non_claude", len(test_nc_recs), ASSEMBLED / "core_test_non_claude.jsonl")
print("test_claude_only", len(test_cl_recs), ASSEMBLED / "core_test_claude_only.jsonl")

lane_f = df["generator_lane"].fillna("(none)")
_om = df["origin_model"].fillna("(none)")
_tb = df["time_band"].fillna("(none)")


def _flatten_crosstab(ct: pd.DataFrame) -> pd.DataFrame:
    t = ct.copy()
    if isinstance(t.columns, pd.MultiIndex):
        t.columns = [
            "__".join(str(x) for x in col) if isinstance(col, tuple) else str(col)
            for col in t.columns
        ]
    return t.reset_index()


def _md_table(flat: pd.DataFrame) -> str:
    cols = [str(c) for c in flat.columns]

    def esc(x: object) -> str:
        return str(x).replace("|", "\\|").replace("\n", " ")

    header = "| " + " | ".join(esc(c) for c in cols) + " |"
    sep = "|" + "|".join([":---"] * len(cols)) + "|"
    rows = []
    for _, row in flat.iterrows():
        rows.append("| " + " | ".join(esc(row[c]) for c in flat.columns) + " |")
    return "\n".join([header, sep] + rows)


def _section(title: str, ct: pd.DataFrame) -> str:
    return "\n".join(["", f"## {title}", "", _md_table(_flatten_crosstab(ct)), ""])


def _save_crosstab_csv(name: str, ct: pd.DataFrame) -> Path:
    p = TABLES_DIR / f"core_crosstab_{name}.csv"
    _flatten_crosstab(ct).to_csv(p, index=False)
    return p


_ct_sf_sp = pd.crosstab(df["scenario_family"], df["split"])
_ct_ch_sp = pd.crosstab(df["channel"], df["split"])
_ct_lb_sp = pd.crosstab(df["label"], df["split"])
_ct_len_sp = pd.crosstab(df["length_bin"], df["split"])
_ct_lane_sp = pd.crosstab(lane_f, df["split"])
_ct_sf_lab = df.groupby(["scenario_family", "label"]).size().unstack(fill_value=0)
_ct_sf_spl_lab = pd.crosstab([df["scenario_family"], df["split"]], df["label"])
_ct_ch_spl_lab = pd.crosstab([df["channel"], df["split"]], df["label"])
_ct_fr_spl_lab = pd.crosstab([df["fraudness"], df["split"]], df["label"])
_ct_len_sf_sp = pd.crosstab([df["length_bin"], df["scenario_family"]], df["split"])
_ct_src_sf_sp = pd.crosstab([df["source_family"], df["scenario_family"]], df["split"])
_ct_om_sf_sp = pd.crosstab([_om, df["scenario_family"]], df["split"])
_ct_lane_sf_sp = pd.crosstab([lane_f, df["scenario_family"]], df["split"])
_ct_tb_sf_sp = pd.crosstab([_tb, df["scenario_family"]], df["split"])
_ct_sf_es_lab = pd.crosstab([df["scenario_family"], df["core_eval_slice"]], df["label"])

_crosstab_csv_paths: list[str] = []
for nm, ct in [
    ("scenario_family__split", _ct_sf_sp),
    ("channel__split", _ct_ch_sp),
    ("label__split", _ct_lb_sp),
    ("length_bin__split", _ct_len_sp),
    ("generator_lane__split", _ct_lane_sp),
    ("scenario_family__label", _ct_sf_lab),
    ("scenario_family__split__label", _ct_sf_spl_lab),
    ("channel__split__label", _ct_ch_spl_lab),
    ("fraudness__split__label", _ct_fr_spl_lab),
    ("length_bin__scenario_family__split", _ct_len_sf_sp),
    ("source_family__scenario_family__split", _ct_src_sf_sp),
    ("origin_model__scenario_family__split", _ct_om_sf_sp),
    ("generator_lane__scenario_family__split", _ct_lane_sf_sp),
    ("time_band__scenario_family__split", _ct_tb_sf_sp),
    ("scenario_family__core_eval_slice__label", _ct_sf_es_lab),
]:
    _crosstab_csv_paths.append(str(_save_crosstab_csv(nm, ct)))

_diag_md_parts = [
    "# Core — split diagnostics",
    "",
    f"Сгенерировано при сборке Core (UTC: {run_ts}). Все таблицы ниже — **факт** по `core_v2.jsonl` после сплита.",
    "",
    "Источник спецификации: `v2/docs/dataset_design_final.md`, `v2/docs/dataset_contract.md`.",
    "",
    "**Политика:** train / val / test_seen балансируются как **human vs seen_llm**; Claude holdout — отдельный add-on только в test.",
    "",
    "## Сводка по строкам",
    "",
    f"- Всего строк: **{len(df)}**",
    f"- train / val / test (split): **{(df['split'] == 'train').sum()}** / **{(df['split'] == 'val').sum()}** / **{(df['split'] == 'test').sum()}**",
    f"- `core_eval_slice`: {df['core_eval_slice'].value_counts().to_dict()}",
    f"- Claude holdout (`generator_lane`): **{int(claude_m.sum())}**",
    "",
    "## Интерпретация test",
    "",
    "- **test_full** (`core_test_full.jsonl`, `core_test.jsonl`): весь `split=test`; операционный union, не единственная главная метрика.",
    "- **test_non_claude** (`core_test_non_claude.jsonl`): только `core_eval_slice=test_seen` — основной сопоставимый test (seen-генераторы).",
    "- **test_claude_only** (`core_test_claude_only.jsonl`): `test_claude_holdout` — stress-test по holdout-генератору.",
    _section("`scenario_family` × `split`", _ct_sf_sp),
    _section("`channel` × `split`", _ct_ch_sp),
    _section("`label` × `split`", _ct_lb_sp),
    _section("`length_bin` × `split`", _ct_len_sp),
    _section("`generator_lane` × `split`", _ct_lane_sp),
    _section("`scenario_family` × `label` (counts)", _ct_sf_lab),
    _section("`scenario_family` × `split` × `label`", _ct_sf_spl_lab),
    _section("`channel` × `split` × `label`", _ct_ch_spl_lab),
    _section("`fraudness` × `split` × `label`", _ct_fr_spl_lab),
    _section("`length_bin` × `scenario_family` × `split`", _ct_len_sf_sp),
    _section("`source_family` × `scenario_family` × `split`", _ct_src_sf_sp),
    _section("`origin_model` × `scenario_family` × `split`", _ct_om_sf_sp),
    _section("`generator_lane` × `scenario_family` × `split`", _ct_lane_sf_sp),
    _section("`time_band` × `scenario_family` × `split`", _ct_tb_sf_sp),
    _section("`scenario_family` × `core_eval_slice` × `label`", _ct_sf_es_lab),
]
PATH_DIAG_MD.write_text("\n".join(_diag_md_parts), encoding="utf-8")
print("Wrote:", PATH_DIAG_MD)

diag_json = {
    "created_utc": run_ts,
    "row_counts": {
        "total": int(len(df)),
        "train": int((df["split"] == "train").sum()),
        "val": int((df["split"] == "val").sum()),
        "test_full": int((df["split"] == "test").sum()),
        "test_seen": int((df["core_eval_slice"] == "test_seen").sum()),
        "test_claude_holdout": int((df["core_eval_slice"] == "test_claude_holdout").sum()),
    },
    "validation_checks": validation["checks"],
    "crosstab_csv": _crosstab_csv_paths,
}
PATH_DIAG_JSON.write_text(json.dumps(diag_json, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", PATH_DIAG_JSON)
print("Crosstab CSVs:", len(_crosstab_csv_paths))

manifest = {
    "created_utc": run_ts,
    "assembly_policy": "core_v2_seen_balanced_plus_claude_holdout",
    "main_seen_balance_policy": "human_vs_seen_llm",
    "claude_holdout_policy": "additional_test_only_slice",
    "claude_holdout_rel_frac": CLAUDE_HOLDOUT_REL_FRAC,
    "main_seen_budget_per_family": main_seen_budget_per_family,
    "claude_holdout_budget_per_family": claude_holdout_budget_per_family,
    "random_seed": RANDOM_SEED,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "test_frac": TEST_FRAC,
    "inputs": {"dataset1_human": str(PATH_DS1), "dataset2_llm": str(PATH_DS2)},
    "outputs": {
        "core_v2": str(PATH_CORE_FULL),
        "core_train": str(ASSEMBLED / "core_train.jsonl"),
        "core_val": str(ASSEMBLED / "core_val.jsonl"),
        "core_test": str(ASSEMBLED / "core_test.jsonl"),
        "core_test_full": str(ASSEMBLED / "core_test_full.jsonl"),
        "core_test_non_claude": str(ASSEMBLED / "core_test_non_claude.jsonl"),
        "core_test_claude_only": str(ASSEMBLED / "core_test_claude_only.jsonl"),
        "core_split_diagnostics_json": str(PATH_DIAG_JSON),
    },
    "row_counts": {
        "total": int(len(df)),
        "train": int((df["split"] == "train").sum()),
        "val": int((df["split"] == "val").sum()),
        "test_full": int((df["split"] == "test").sum()),
        "test_non_claude": int((df["core_eval_slice"] == "test_seen").sum()),
        "test_claude_only": int((df["core_eval_slice"] == "test_claude_holdout").sum()),
    },
    "claude_holdout_rows": int(claude_m.sum()),
    "split_diagnostics": {
        "markdown": str(PATH_DIAG_MD),
        "json": str(PATH_DIAG_JSON),
        "crosstab_csv_glob": str(TABLES_DIR / "core_crosstab_*.csv"),
    },
    "downsampling": {
        "target_n_per_side_spec": {k: v for k, v in TARGET_N_PER_SIDE.items()},
        "fraud_sms_max_per_side": FRAUD_SMS_MAX_PER_SIDE,
        "legitimate_sms_target_cap": LEGITIMATE_SMS_TARGET_CAP,
        "target_pairs_financial_qa": TARGET_PAIRS_FQ,
        "main_seen_balance_policy": "human_vs_seen_llm",
        "claude_holdout_policy": "additional_test_only_slice",
        "claude_holdout_rel_frac": CLAUDE_HOLDOUT_REL_FRAC,
        "main_seen_budget_per_family": main_seen_budget_per_family,
        "claude_holdout_budget_per_family": claude_holdout_budget_per_family,
        "seen_llm_split_quota_rule": "openai_mistral_proportional_to_raw_volume_then_stratified_length_bin",
        "raw_input_counts": raw_counts,
        "report": downsampling_report,
    },
    "validation": validation,
    "assembly_notes": manifest_log,
}
PATH_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote manifest:", PATH_MANIFEST)


In [ ]:
TARGETS_72 = {
    "phishing_email": ((2500, 3000), (2500, 3000)),
    "advance_fee_scam_email": ((1200, 1800), (1200, 1800)),
    "fraud_sms_deceptive": ((1200, 1800), (1200, 1800)),
    "legitimate_email": ((2500, 3000), (2500, 3000)),
    "legitimate_sms": ((800, 1200), (800, 1200)),
    "financial_qa": ((800, 1200), (800, 1200)),
}


def in_range(n: int, lo: int, hi: int) -> str:
    return "внутри §7.2" if lo <= n <= hi else "вне §7.2"


counts_h = df[df["label"] == 0].groupby("scenario_family").size()
counts_l = df[df["label"] == 1].groupby("scenario_family").size()
counts_seen = df[(df["label"] == 1) & (~claude_m)].groupby("scenario_family").size()

lines = [
    "# Core v2 — описание итогового датасета",
    "",
    f"Сгенерировано из `19_core_train_val_test_assembly.ipynb` (UTC: {run_ts}).",
    "",
    "## Источник истины и политика сборки",
    "",
    "Спецификация: `v2/docs/dataset_design_final.md`, `v2/docs/dataset_contract.md`. Сборка **не** меняет список `scenario_family`, источники, preprocessing, парную логику `financial_qa` и правило **Claude только в test**.",
    "",
    "### Что исправлено относительно предыдущей сборки",
    "",
    "- Раньше family-level баланс фактически считался как human ↔ (seen_llm + Claude holdout); после изъятия Claude в `test_claude_only` основные сплиты (train/val/test_seen) становились **human-heavy**.",
    "- Теперь **train, val, test_seen** балансируются как **human ↔ seen_llm** с бюджетом `main_seen_budget_per_side = min(target_cap, n_human_raw, n_seen_llm_raw)`. Claude **не** участвует в этом бюджете.",
    "- **Claude holdout** — отдельный add-on: `claude_holdout_budget = min(round(main_seen_budget * CLAUDE_HOLDOUT_REL_FRAC), n_claude_raw)` (дефолт **0.25**), только `core_eval_slice=test_claude_holdout`.",
    "",
    "## Два смысла баланса на уровне семьи",
    "",
    "- **Main comparable balance:** human ↔ seen_llm в train/val/test_seen (и в сумме main pool).",
    "- **Дополнительный срез:** Claude holdout добавляет LLM-строки только в `test_claude_only`, без участия в бюджете main.",
    "",
    f"- `assembly_policy`: **core_v2_seen_balanced_plus_claude_holdout** (см. `core_manifest.json`).",
    "",
    "## Цель",
    "",
    "Стратифицированный downsampling и **симметричный** сплит 75/15/10 для main pool (human + seen_llm по каждой семье); holdout Claude только в test. `financial_qa` — только полные пары (bilateral seen-only control).",
    "",
    "## Downsampling",
    "",
    "1. **Human (main):** до `main_seen_budget_per_side` со стратами `length_bin` + при необходимости `source_family` (legitimate / fraud SMS), `time_band` (phishing / advance fee).",
    "2. **Seen LLM (main):** до того же бюджета; квота OpenAI vs Mistral **пропорционально сырому объёму**, внутри — стратификация по `length_bin` (см. `manifest.downsampling.seen_llm_split_quota_rule`).",
    f"3. **Claude holdout:** отдельно до `claude_holdout_budget`, страта `length_bin`; параметр `CLAUDE_HOLDOUT_REL_FRAC={CLAUDE_HOLDOUT_REL_FRAC}`.",
    "4. **`financial_qa`:** только полные пары `question_id`; выбор до `TARGET_PAIRS_FQ` со стратификацией пар по `length_bin` human-стороны. Claude для этой семьи не используется.",
    "",
    f"- Исходные объёмы: dataset1 **{raw_counts['dataset1_total']}** строк, dataset2 **{raw_counts['dataset2_total']}** строк.",
    f"- После отбора: **{manifest['row_counts']['total']}** строк.",
    "",
    "## Метод сплита",
    "",
    f"- Seed `{RANDOM_SEED}`; доли **main pool**: {int(TRAIN_FRAC * 100)}% / {int(VAL_FRAC * 100)}% / {int(TEST_FRAC * 100)}% для **каждой** non-FQ семьи симметрично на human и seen_llm (одинаковые целочисленные квоты train/val/test).",
    "- Страты human: `human_strata_cols`; seen_llm: `length_bin` + `generator_lane` (+ `origin_model` при вариативности).",
    "- Все строки Claude → `split=test`, `core_eval_slice=test_claude_holdout` (не участвуют в train/val/test_seen).",
    "",
    "## Интерпретация test",
    "",
    "- **`test_non_claude`** (`core_test_non_claude.jsonl`, `test_seen`) — **основной сопоставимый** test (human vs seen LLM).",
    "- **`test_claude_only`** — **unseen-generator / holdout** stress-test.",
    "- **`test_full`** — объединение test_seen и Claude holdout; **операционный** полный test, может быть LLM-heavy из-за Claude; не подменяет интерпретацию основных сплитов.",
    "",
    "## Эксперименты и текст ВКР (жёсткий протокол)",
    "",
    "Главные метрики показывать **отдельно** для **val**, **test_non_claude**, **test_claude_only**. **`test_full`** использовать только как **дополнительный** агрегированный срез; **не** делать его единственной главной цифрой качества.",
    "",
    "В ВКР явно зафиксировать: main comparable pool **balanced human vs seen_llm**; Claude — **отдельный holdout stress-test**; Core — **multi-genre** бенчмарк; есть **temporal bias** и ограничения покрытия генераторов; срез **SMS fraud** **data-limited**. Подробнее: `v2/docs/project_status.md`, `v2/docs/thesis_constraints.md`.",
    "",
    "## Объёмы по split",
    "",
    f"| область | count |",
    "|---------|------:|",
    f"| train | {manifest['row_counts']['train']} |",
    f"| val | {manifest['row_counts']['val']} |",
    f"| test_full | {manifest['row_counts']['test_full']} |",
    f"| test_non_claude (test_seen) | {manifest['row_counts']['test_non_claude']} |",
    f"| test_claude_only | {manifest['row_counts']['test_claude_only']} |",
    f"| **total** | **{manifest['row_counts']['total']}** |",
    "",
    f"- Строк с `generator_lane == holdout_claude`: **{manifest['claude_holdout_rows']}**.",
    "",
]

_split_order = ["train", "val", "test"]
_ct_ch_sp = (
    pd.crosstab(df["channel"], df["split"])
    .reindex(columns=_split_order, fill_value=0)
    .astype(int)
)
_ct_sc_sp = (
    pd.crosstab(df["scenario_family"], df["split"])
    .reindex(columns=_split_order, fill_value=0)
    .astype(int)
)


def _lane_bucket(r: pd.Series) -> str:
    if int(r["label"]) == 0:
        return "(human)"
    gl = r["generator_lane"]
    if pd.notna(gl) and str(gl).strip():
        return str(gl)
    om = r["origin_model"]
    if pd.notna(om) and str(om).strip():
        return f"llm:{om}"
    return "llm:(unspecified)"


_gb = df.apply(_lane_bucket, axis=1)
_ct_sc_gb = pd.crosstab(df["scenario_family"], _gb)


def _sort_gen_cols(cols):
    def _key(x):
        xs = str(x)
        if xs == "(human)":
            return (0, xs)
        if xs == "holdout_claude":
            return (1, xs)
        if xs.startswith("llm:"):
            return (2, xs)
        return (3, xs)

    return sorted(cols, key=_key)


_ct_sc_gb = _ct_sc_gb[_sort_gen_cols(_ct_sc_gb.columns)]
_test_df = df[df["split"] == "test"]
_ct_test = pd.crosstab(_test_df.apply(_lane_bucket, axis=1), _test_df["label"]).rename(
    columns={0: "human (label=0)", 1: "llm (label=1)"}
)


def _md_ct(ct: pd.DataFrame, quote_row: bool) -> list[str]:
    cols = list(ct.columns)
    hcells = []
    for c in cols:
        s = str(c)
        hcells.append(f"`{s}`" if any(ch in s for ch in " |") or ":" in s else s)
    out = [
        "| | " + " | ".join(hcells) + " | **Σ** |",
        "|---|" + "|".join([":---" for _ in cols]) + "|---:|",
    ]
    for idx in ct.index:
        ir = str(idx).replace("|", "\\|")
        row0 = f"`{ir}`" if quote_row else ir
        vals = [int(ct.loc[idx, c]) for c in cols]
        tot = sum(vals)
        out.append(f"| {row0} | " + " | ".join(str(v) for v in vals) + f" | **{tot}** |")
    return out


_llm_total = int((df["label"] == 1).sum())
_claude_total = int((df["generator_lane"] == "holdout_claude").sum())
_frac = 100.0 * _claude_total / _llm_total if _llm_total else 0.0
_test_n = int(validation["test_counts"]["test_full"])
_test_h = int((_test_df["label"] == 0).sum())
_test_l = int((_test_df["label"] == 1).sum())
try:
    _tcl = int(_ct_test.loc["holdout_claude", "llm (label=1)"])
except (KeyError, TypeError):
    _tcl = 0
_t_other_llm = _test_l - _tcl

lines += [
    "## Распределение по `channel` × `split`",
    "",
    *_md_ct(_ct_ch_sp, quote_row=True),
    "",
    "## Распределение по `scenario_family` × `split`",
    "",
    *_md_ct(_ct_sc_sp, quote_row=True),
    "",
    "## Генерация по `scenario_family` (модель / lane)",
    "",
    "Для **human** (`label=0`) — `(human)`. Для **LLM** — `generator_lane`, иначе `llm:` + `origin_model` (часто для `financial_qa`).",
    "",
    *_md_ct(_ct_sc_gb, quote_row=True),
    "",
    "## Множество `test`",
    "",
    *_md_ct(_ct_test, quote_row=True),
    "",
    f"- В **test** (**{_test_n}** строк): human **{_test_h}**, LLM **{_test_l}**; из LLM в test Claude holdout **{_tcl}**, прочие LLM **{_t_other_llm}**.",
    f"- Среди **всех** LLM в Core: Claude holdout **{_claude_total} / {_llm_total} ≈ {_frac:.1f}%**.",
    "",
    "## Проверки",
    "",
]
for c in validation["checks"]:
    st = "PASS" if c["ok"] else "FAIL"
    lines.append(f"- **{c['name']}**: {st} — `{json.dumps({k: v for k, v in c.items() if k != 'name'}, ensure_ascii=False)}`")
lines += [
    "",
    "## Сопоставление с §7.2 (human vs все LLM в Core)",
    "",
    "| scenario_family | human | all_llm | seen_llm (excl. Claude) | §7.2 human | §7.2 llm |",
    "|----------------|------:|--------:|-------------------------|------------|----------|",
]
for sf in TARGETS_72:
    h = int(counts_h.get(sf, 0))
    l = int(counts_l.get(sf, 0))
    s = int(counts_seen.get(sf, 0))
    (hlo, hhi), (llo, lhi) = TARGETS_72[sf]
    lines.append(f"| `{sf}` | {h} | {l} | {s} | {hlo}–{hhi} | {llo}–{lhi} |")

lines += [
    "",
    "## Ограничения",
    "",
    "- Human `fraud_sms_deceptive` ≤ ~852 строк: ниже середины §7.2 (доступность данных).",
    "- `legitimate_email`: при цели 3000/сторона на main human↔seen_llm факт может быть **ниже**, если узкое место — **seen**-LLM пул в dataset2 (см. `downsampling.report.per_family.legitimate_email`).",
    "- `legitimate_sms`: объём ограничен доступностью human/seen_llm.",
    "- Выводы не распространять на произвольный unseen generator beyond заявленных seen + Claude holdout.",
    "",
    "## Артефакты",
    "",
    "- `v2/data/interim/assembled/core_v2.jsonl`, `core_train.jsonl`, `core_val.jsonl`, `core_test.jsonl` (= test_full), `core_test_full.jsonl`, `core_test_non_claude.jsonl`, `core_test_claude_only.jsonl`",
    "- `v2/data/interim/assembled/core_manifest.json`, `core_split_diagnostics.json`",
    "- `v2/docs/core_split_diagnostics.md`",
    "- `v2/outputs/tables/core_crosstab_*.csv`",
    "",
]

PATH_DESC.write_text("\n".join(lines), encoding="utf-8")
print("Wrote:", PATH_DESC)
